Human in the loop is a design approach in the AI systems where a human actiively participates at critical point of the AI workflow. either to supervise, approve, correct or guide the models output.

->its like putting a human check point inside AI pipeline so that important decisions are not made by the model but also human intervention.

->HITL ensures 
.accuracy
.safety
.Ethical Alignment
.better user experience


Human-in-the-Loop allows a workflow to pause execution and wait for human approval, feedback, or correction before continuing.

Instead of letting the agent make every decision autonomously, you insert checkpoints where a human can review what the AI wants to do

->Checkpointing means saving the graph's state during execution so it can be resumed later.



Start
  ↓
Generate Response
  ↓
Approval Node
  ↓
Send Response

Parallel Fan output:
->Instead of doing tasks parallely.
           ┌── Task A
Start ─────┤
           ├── Task B
           │
           └── Task C

In [72]:
# from typing import Annotated
from langchain_core.messages import AnyMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [2]:
import os
from dotenv import load_dotenv

load_dotenv("myenv.env")

# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


In [3]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [73]:
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })

    approved = decision["approved"].strip().lower()

    if approved == "yes":
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    elif approved == "no":
        return {
            "messages": [
                AIMessage(content="Not approved.")
            ]
        }

    else:
        return {
            "messages": [
                AIMessage(
                    content="Invalid input. Please enter only yes or no."
                )
            ]
        }

In [ ]:
# def chat_node(state: ChatState):

#     decision = interrupt({
#         "type": "approval",
#         "reason": "Model is about to answer a user question.",
#         "question": state["messages"][-1].content,
#         "instruction": "Approve this question? yes/no"
#     })
    
#     if decision["approved"] == 'no':
#         return {"messages": [AIMessage(content="Not approved.")]}

#     else:
#         response = llm.invoke(state["messages"])
#         return {"messages": [response]}

In [86]:
#  3. Build the graph: START -> chat -> END(for understanding)
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [8]:
print(app)

In [87]:
#  when checkpoint are used config and threads has to be used to name or get the information
config = {"configurable": {"thread_id": '104'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain Deep learning in an easy way.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [88]:
print(result)

{'messages': [HumanMessage(content='Explain Deep learning in an easy way.', additional_kwargs={}, response_metadata={}, id='9852dd9d-ca9e-49e6-b6d3-5beb89702b9b')], '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain Deep learning in an easy way.', 'instruction': 'Approve this question? yes/no'}, id='c0f1d9e4e3e27bb0e1cbb4abe575159e')]}


In [89]:
message = result['__interrupt__'][0].value
# print(message)
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain Deep learning in an easy way.',
 'instruction': 'Approve this question? yes/no'}

In [90]:
user = input(f"\nBackend message - {message} \n Approve this question? (yes/no): ")

In [91]:
# Resume the graph with the approval decision
final_result = app.invoke(
    Command(resume={"approved": user}),
    config=config,
)


In [92]:
print(final_result["messages"][-1].content)

Not approved.


In [61]:
#  3. Build the graph: START -> chat -> END(for understanding)
builder1 = StateGraph(ChatState)

builder1.add_node("chat", chat_node)

builder1.add_edge(START, "chat")
builder1.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer1 = MemorySaver()

# Compile the app
app1 = builder1.compile(checkpointer=checkpointer1)

In [77]:
#  when checkpoint are used config and threads has to be used to name or get the information
config = {"configurable": {"thread_id": '1235'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain AI integration in real world?.")
    ]
}

# Invoke the graph for the first time
result = app1.invoke(initial_input, config=config)

In [78]:
message=result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain AI integration in real world?.',
 'instruction': 'Approve this question? yes/no'}

In [79]:
user1= input(f"backend message- {message}\n type yes/no ?")

In [ ]:
final_ans= app1.invoke(
    Command(resume={"approved": user1}),
    config=config,
)

In [81]:
print(final_ans["messages"][-1].content)

AI integration in the real world refers to the implementation of artificial intelligence technologies into various sectors and aspects of everyday life to enhance efficiency, productivity, and decision-making processes. Here are some key areas where AI integration is making a significant impact:

### 1. Healthcare
- **Diagnosis and Treatment**: AI algorithms analyze medical images and data to assist doctors in diagnosing diseases more accurately and quickly. For example, AI can help in detecting tumors in mammograms or identifying abnormalities in MRI scans.
- **Drug Discovery**: AI accelerates the drug discovery process by predicting how new drugs will interact with the human body, reducing the time and cost of bringing new medications to market.
- **Personalized Medicine**: AI analyzes genetic information to tailor treatments to individual patients, improving the effectiveness of therapies.

### 2. Finance
- **Fraud Detection**: AI systems monitor transactions in real-time to detect 